In [1]:
import pandas as pd
import numpy as np
from src.processing import Processing
import config
import matplotlib.pyplot as plt
from itertools import cycle

from sklearn import svm, datasets
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score

In [2]:
# Import some data to play with
iris = datasets.load_iris()
X = iris.data
y = iris.target

# Binarize the output
y = label_binarize(y, classes=[0, 1, 2])
n_classes = y.shape[1]

# Add noisy features to make the problem harder
random_state = np.random.RandomState(0)
n_samples, n_features = X.shape
X = np.c_[X, random_state.randn(n_samples, 200 * n_features)]

# shuffle and split training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=0)

print(n_classes)

# Learn to predict each class against the other
classifier = OneVsRestClassifier(
    svm.SVC(kernel="linear", probability=True, random_state=random_state)
)
y_score = classifier.fit(X_train, y_train).decision_function(X_test)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

3


In [3]:
list_of_numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

list_of_numbers[8:15]

[9, 10]

In [3]:
features_orig = pd.read_csv(config.RADIOMICS_FINAL_FEATURES_NO_FILTERS)
# Read header
header = features_orig.columns

In [9]:
features_all = pd.read_csv(config.RADIOMICS_FINAL_FEATURES)
# Read header
header = features_all.columns

with open("features_all.txt", "w") as f:
    # Write as a set with double quotes
    f.write(str(set(header)))
    f.write("\n")
    

In [7]:
list(range(5))

[0, 1, 2, 3, 4]

In [6]:
train = [1,2,3,4,5,6,7,8,9,10]
test = [11,12,13,14,15,16,17,18,19,20]

train_2 = [21,22,23,24,25,26,27,28,29,30]
test_2 = [31,32,33,34,35,36,37,38,39,40]

# Indexes of the train and test sets
a = [([0, 1, 2, 3, 4], [5])]

# Indexes of the train and test sets 2
b = [([1], [1])]

a.append(b[0])

print(a)

[([0, 1, 2, 3, 4], [5]), ([1], [1])]


In [ ]:
train = [1,2,3,4,5,6,7,8,9,10]
test = [11,12,13,14,15,16,17,18,19,20]

train_2 = [21,22,23,24,25,26,27,28,29,30]
test_2 = [31,32,33,34,35,36,37,38,39,40]

# Indexes of the train and test sets
a = [([0, 1, 2, 3, 4], [5])]

# Indexes of the train and test sets 2
b = [([1], [1])]

# Include b[0] in a[0] and b[1] in a[1] taking into account that the indexes are relative to the original data
a[0][0].extend([i + len(train) for i in b[0][0]])  # Añade el índice relativo de 'train_2'
a[0][1].extend([i + len(test) for i in b[0][1]])    # Añade el índice relativo de 'test_2'

print(a)

[([0, 1, 2, 3, 4, 11], [5, 11])]


In [3]:
# Create false df with 5 cols and 3 rows
df = pd.DataFrame(np.random.randint(0,100,size=(3, 5)), columns=list('ABCDE'))

# Add a column with mouse_id
df['m_id'] = ['a', 'a', 'b']

for mouse_id, group in df.groupby('m_id'):
    print(group)
    print(mouse_id)
    print('---')

    A   B   C   D   E m_id
0  68  10  77  87  39    a
1  58  63  51  55  98    a
a
---
    A   B   C   D   E m_id
2  90  81  82  68  93    b
b
---


In [8]:
dataclass = Processing(source_path=config.RADIOMICS_FEATURES_3_GROUPS)
dataclass.read()
df = dataclass.preprocess(remove_mice_id=False, start_col=3)
df.sort_values(by=['m_id', 'day_of_study'], inplace=True)

# Save as csv
df.to_csv('input/features/radiomics_features_all_3_groups_sorted.csv', index=False)


In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit, TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score, precision_recall_curve, auc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

# Step 1: Load and preprocess the data
dataclass = Processing(source_path=config.RADIOMICS_FINAL_FEATURES_NO_FILTERS, all_features=False)
dataclass.read()
df = dataclass.preprocess()  # Keep mouse ID for the later splits

# Save the preprocessed data to a file
df.to_csv('preprocessed.csv', index=False)


# Hyperparameter grid for XGBClassifier
param_grid = {
    'n_estimators': [100, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1.0]
}


from sklearn.model_selection import TimeSeriesSplit
from imblearn.over_sampling import SMOTE
from collections import Counter

def expanding_window_split_with_oversampling(df, n_splits=4):
    """
    Perform Expanding Window Split with Oversampling.

    Args:
        df (pd.DataFrame): DataFrame containing all instances, sorted by time.
        n_splits (int): Number of splits for expanding windows.

    Returns:
        splits (dict): Dictionary with train-test-validation splits for each fold.
    """
    splits = {}

    # Sort the DataFrame by 'day_of_study' to ensure chronological order
    df = df.sort_values(by='day_of_study').reset_index(drop=True)

    # TimeSeriesSplit for expanding window
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for i, (train_indices, test_indices) in enumerate(tscv.split(df)):
        test_set_size = len(test_indices)
        
        # Validation indices (last part of the training set)
        val_indices = train_indices[-test_set_size:]
        
        # Prepare training, validation, and testing sets
        X_train = df.iloc[train_indices].drop(columns=['group_name']).reset_index(drop=True)
        y_train = df.iloc[train_indices]['group_name'].reset_index(drop=True)
        
        y_val = df.iloc[val_indices]['group_name'].reset_index(drop=True)
        
        X_test = df.iloc[test_indices].drop(columns=['group_name']).reset_index(drop=True)
        y_test = df.iloc[test_indices]['group_name'].reset_index(drop=True)

        # Check if training set has both classes; apply oversampling if not
        # if len(y_train.unique()) < 2:
        #     print(f"Before oversampling, y_train distribution: {Counter(y_train)}")
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print(f"After oversampling, y_train distribution: {Counter(y_train)}")

        # Check if validation set has both classes
        if len(y_val.unique()) < 2:
            print(f"Validation set in fold {i} does not contain both classes. Consider adjusting validation strategy.")

        # Store splits
        splits[i] = {
            'X_train': X_train,
            'y_train': y_train,
            'validation_indices': val_indices,
            'y_val': y_val,
            'X_test': X_test,
            'y_test': y_test
        }
    
    return splits

# Apply expanding window split with oversampling
m_data = expanding_window_split_with_oversampling(df)

# Function to create PredefinedSplit from the cv indices
def create_predefined_split(n_samples:int, cv_indices):
    test_fold = np.full((n_samples,), -1)
    test_fold[cv_indices] = 0
    ps = PredefinedSplit(test_fold)
    return ps

from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

# Track results for each fold
fold_accuracies, fold_f1_macros, fold_precisions, fold_recalls, fold_roc_aucs, fold_pr_aucs, fold_f1_micros, fold_f1_weighted = [], [], [], [], [], [], [], []
class_metrics = {'true_positives': [], 'false_negatives': [], 'false_positives': [], 'true_negatives': []}

# Store ROC and PR curve points for all folds
roc_curves = []
pr_curves = []

# Train and evaluate
for fold, split_data in m_data.items():

    print(f"Fold {fold}:")
    print(f"Train class distribution: {Counter(split_data['y_train'])}")
    print(f"Validation class distribution: {Counter(split_data['y_val'])}")
    print(f"Test class distribution: {Counter(split_data['y_test'])}")

    # Define PredefinedSplit
    ps = create_predefined_split(len(split_data['X_train']), split_data['validation_indices'])
    
    # Grid search for hyperparameter tuning
    model = xgb.XGBClassifier(random_state=42)
    grid_search = GridSearchCV(model, param_grid, cv=ps, scoring='f1_macro', n_jobs=-1)
    grid_search.fit(split_data['X_train'], split_data['y_train'])

    best_model = grid_search.best_estimator_
    print(f"Best parameters for fold {fold}: {grid_search.best_params_}")

    # Evaluate on the test set
    preds = best_model.predict(split_data['X_test'])
    preds_proba = best_model.predict_proba(split_data['X_test'])[:, 1]  # Probability for the positive class
    
    # Calculate metrics
    accuracy = accuracy_score(split_data['y_test'], preds)
    f1_macro = f1_score(split_data['y_test'], preds, average='macro')
    f1_micro = f1_score(split_data['y_test'], preds, average='micro')
    f1_weighted = f1_score(split_data['y_test'], preds, average='weighted')
    precision = precision_score(split_data['y_test'], preds)
    recall = recall_score(split_data['y_test'], preds)

    fpr, tpr, _ = roc_curve(split_data['y_test'], preds_proba, pos_label=1)
    roc_auc = auc(fpr, tpr)

    precision_c, recall_c, _ = precision_recall_curve(split_data['y_test'], preds_proba, pos_label=1)
    pr_auc = average_precision_score(split_data['y_test'], preds_proba)
    
    roc_curves.append((fpr, tpr))
    pr_curves.append((precision_c, recall_c))

    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(split_data['y_test'], preds).ravel()

    # Print fold results
    print(f"\nFold {fold} results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-macro: {f1_macro:.4f}")
    print(f"F1-micro: {f1_micro:.4f}")
    print(f"F1-weighted: {f1_weighted:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"PR AUC: {pr_auc:.4f}")
    print(f"True Positives: {tp}, False Negatives: {fn}, False Positives: {fp}, True Negatives: {tn}")

    # Track results
    fold_accuracies.append(accuracy)
    fold_f1_macros.append(f1_macro)
    fold_f1_micros.append(f1_micro)
    fold_f1_weighted.append(f1_weighted)
    fold_precisions.append(precision)
    fold_recalls.append(recall)
    fold_roc_aucs.append(roc_auc)
    fold_pr_aucs.append(pr_auc)

    class_metrics['true_positives'].append(tp)
    class_metrics['false_negatives'].append(fn)
    class_metrics['false_positives'].append(fp)
    class_metrics['true_negatives'].append(tn)

# Plotting ROC and Precision-Recall curves side by side
plt.figure(figsize=(14, 6))

# ROC Curve Plot
plt.subplot(1, 2, 1)  # First plot (1 row, 2 columns, 1st plot)
for i, (fpr, tpr) in enumerate(roc_curves):
    plt.plot(fpr, tpr, label=f'Fold {i} (ROC AUC = {fold_roc_aucs[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label="Random Classifier")
plt.title("ROC Curve Across Folds")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")

# Precision-Recall Curve Plot
plt.subplot(1, 2, 2)  # Second plot (1 row, 2 columns, 2nd plot)
for i, (precision, recall) in enumerate(pr_curves):
    plt.plot(recall, precision, label=f'Fold {i} (PR AUC = {fold_pr_aucs[i]:.2f})')
plt.title("Precision-Recall Curve Across Folds")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="lower right")

# Display the plots side by side
plt.tight_layout()  # Adjust layout for better spacing
plt.show()

# Display summary statistics across folds
print(f"\nSummary Across Folds:")
print(f"Mean accuracy: {np.mean(fold_accuracies):.4f}")
print(f"Mean F1-macro: {np.mean(fold_f1_macros):.4f}")
print(f"Mean F1-micro: {np.mean(fold_f1_micros):.4f}")
print(f"Mean F1-weighted: {np.mean(fold_f1_weighted):.4f}")
print(f"Mean precision: {np.mean(fold_precisions):.4f}")
print(f"Mean recall: {np.mean(fold_recalls):.4f}")
print(f"Mean ROC AUC: {np.mean(fold_roc_aucs):.4f}")
print(f"Mean PR AUC: {np.mean(fold_pr_aucs):.4f}")